# RNN, LSTM, and GRU sequence classification

Compare recurrent layers on a small task: identify whether a noisy sequence increases or decreases. We split entire independently generated sequences, select a model on validation loss, and evaluate that model on a held-out test set.

**Prerequisites:** the PyTorch training lesson and tensor shapes. **Environment:** base requirements plus CPU PyTorch. **Execution:** restart and run all cells. No external files or GPU are required. These signals are not sign-language or financial data.


In [ ]:
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
DEVICE = torch.device('cpu')
print('PyTorch:', torch.__version__, 'Device:', DEVICE)


## Generate and split independent sequences

The random split is suitable here because each sequence is generated independently. Real overlapping time windows, related subjects, or chronological forecasting data need grouped or chronological splits to avoid leakage.


In [ ]:
rng = np.random.default_rng(SEED)
samples, steps = 400, 24
y = np.repeat([0, 1], samples // 2)
time = np.linspace(-1, 1, steps, dtype=np.float32)
direction = (2 * y - 1).astype(np.float32)
amplitude = rng.uniform(0.4, 1.2, size=samples).astype(np.float32)
offset = rng.uniform(-0.3, 0.3, size=samples).astype(np.float32)
X = (offset[:, None] + direction[:, None] * amplitude[:, None] * time
     + rng.normal(0, 0.15, (samples, steps))).astype(np.float32)[..., None]
X_development, X_test, y_development, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)
X_train, X_validation, y_train, y_validation = train_test_split(
    X_development, y_development, test_size=0.2, stratify=y_development, random_state=SEED
)
assert X_train.shape[1:] == (steps, 1)
print('Shape: (samples, time steps, features) =', X.shape)
fig, ax = plt.subplots(figsize=(6, 3))
for label, index in [(0, 0), (1, samples // 2)]:
    ax.plot(time, X[index, :, 0], label=['decreasing', 'increasing'][label])
ax.set(xlabel='Time', ylabel='Signal', title='Independent synthetic sequences')
ax.legend()
plt.show()


## Use the final recurrent output for classification

With `batch_first=True`, the recurrent input and output start with the batch dimension. An LSTM also returns cell state; a GRU and a basic RNN return hidden state. This classifier uses the output at the last time step, so the same code supports all three. Sequence length is fixed; variable-length data needs padding/masking or packed sequences.


In [ ]:
class SequenceClassifier(nn.Module):
    def __init__(self, family, hidden=12):
        super().__init__()
        layer = {'RNN': nn.RNN, 'LSTM': nn.LSTM, 'GRU': nn.GRU}[family]
        self.recurrent = layer(input_size=1, hidden_size=hidden, batch_first=True)
        self.classifier = nn.Linear(hidden, 2)
    def forward(self, sequences):
        outputs, state = self.recurrent(sequences)
        return self.classifier(outputs[:, -1, :])

probe = torch.tensor(X_train[:4])
for family in ['RNN', 'LSTM', 'GRU']:
    candidate = SequenceClassifier(family)
    assert candidate(probe).shape == (4, 2)
    print(family, 'parameters:', sum(parameter.numel() for parameter in candidate.parameters()))


## Train on training data and select on validation data

Each batch clears accumulated gradients, computes the loss, backpropagates, and updates parameters. `train()` enables training behavior; `eval()` and `no_grad()` are used for evaluation. The helper below retains a **copy** of the weights with the lowest validation loss. The test set is not used to choose an epoch or model.

Seeds make this small CPU experiment repeatable in the same environment. Bitwise equality across different library versions or hardware is not promised.


In [ ]:
def loader(features, targets, shuffle=False, batch_size=64):
    dataset = TensorDataset(torch.as_tensor(features, dtype=torch.float32),
                            torch.as_tensor(targets, dtype=torch.long))
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=0,
                      generator=torch.Generator().manual_seed(SEED))

def evaluate_classifier(model, batches):
    model.eval()  # disables dropout and uses inference behavior
    total_loss, total, correct = 0.0, 0, 0
    all_predictions = []
    with torch.no_grad():  # validation does not build a gradient graph
        for features, targets in batches:
            logits = model(features.to(DEVICE))
            targets = targets.to(DEVICE)
            total_loss += nn.functional.cross_entropy(logits, targets, reduction='sum').item()
            prediction = logits.argmax(dim=1)
            correct += (prediction == targets).sum().item()
            total += targets.numel()
            all_predictions.append(prediction.cpu().numpy())
    return total_loss / total, correct / total, np.concatenate(all_predictions)

def fit_classifier(model, train_batches, validation_batches, epochs=10, learning_rate=0.005):
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    history, best_state, best_validation_loss = [], None, float('inf')
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, total = 0.0, 0
        for features, targets in train_batches:
            features, targets = features.to(DEVICE), targets.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = nn.functional.cross_entropy(model(features), targets)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * targets.numel()
            total += targets.numel()
        validation_loss, validation_accuracy, _ = evaluate_classifier(model, validation_batches)
        history.append({'epoch': epoch, 'train_loss': total_loss / total,
                        'validation_loss': validation_loss,
                        'validation_accuracy': validation_accuracy})
        if validation_loss < best_validation_loss:
            best_validation_loss = validation_loss
            best_state = copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    model.eval()
    return pd.DataFrame(history)


In [ ]:
models, histories, rows = {}, {}, []
validation_batches = loader(X_validation, y_validation)
for family in ['RNN', 'LSTM', 'GRU']:
    torch.manual_seed(SEED)
    model = SequenceClassifier(family).to(DEVICE)
    history = fit_classifier(model, loader(X_train, y_train, shuffle=True),
                             validation_batches, epochs=8, learning_rate=0.008)
    validation_loss, validation_accuracy, _ = evaluate_classifier(model, validation_batches)
    models[family], histories[family] = model, history
    rows.append({'family': family, 'validation_loss': validation_loss,
                 'validation_accuracy': validation_accuracy})
results = pd.DataFrame(rows).set_index('family')
display(results)
chosen_family = results['validation_loss'].idxmin()
print('Chosen by validation loss:', chosen_family)
fig, ax = plt.subplots(figsize=(6, 3))
for family, history in histories.items():
    ax.plot(history['epoch'], history['validation_loss'], label=family)
ax.set(xlabel='Epoch', ylabel='Validation loss')
ax.legend()
plt.show()


## Evaluate only the selected family on test data

The comparison is illustrative: one synthetic task, one initialization, and small architectures cannot rank recurrent model families generally. Test results are displayed after selection and are not used to change the selected family.


In [ ]:
test_loss, test_accuracy, predictions = evaluate_classifier(
    models[chosen_family], loader(X_test, y_test)
)
print('Held-out', chosen_family, 'loss:', round(test_loss, 4), 'accuracy:', round(test_accuracy, 4))
print(classification_report(y_test, predictions, target_names=['decreasing', 'increasing'], zero_division=0))


## Practice

Increase sequence length and noise using training/validation data. Repeat with several seeds and report variation. For landmark recognition research, continue in the separate [sign-language collection](https://github.com/Muhammad-Huzifa/sign-language-recognition), whose real-data inputs and validation needs are different.

**References:** [RNN](https://docs.pytorch.org/docs/stable/generated/torch.nn.RNN.html), [LSTM](https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html), [GRU](https://docs.pytorch.org/docs/stable/generated/torch.nn.GRU.html).
